In [10]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from cartopy.util import add_cyclic_point
from pathlib import Path
import xcdat as xc
import cmasher as cmr
import xarray as xr
import sys

sys.path.append('../../functions/')
from lag_linregress import *
from monthly_departures import *
from MCA import *


In [11]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

def plot_map(lat, lon, data):
    fig, ax = plt.subplots(subplot_kw={"projection": ccrs.PlateCarree(central_longitude=210)})
    im = ax.pcolormesh(lon, lat, data, shading="auto", transform=ccrs.PlateCarree())
    ax.coastlines()
    fig.colorbar(im, ax=ax)
    plt.show()
    return fig, ax

In [12]:
# Reload edited Python modules automatically before running subsequent cells.
%load_ext autoreload
%autoreload 2

import sys

# Add the project folder only once, even if this cell is rerun.
functions_path = "/scratch/leiff/MCA/amip_clean/project_functions/"
if functions_path not in sys.path:
    sys.path.append(functions_path)

from MCA_cov import *
from significance import *
from plotting import *
from diagnostics import *
# from taylor_setup import *
from setup_diagnostic import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
# Little janky but this block gives us important common info
filepath_output_era5    = '/scratch/leiff/MCA/amip_clean/data/ERA5/tas_194001-202412_g025.nc'
ERA5                    = xc.open_dataset(filepath_output_era5).sel(time=slice('2000-03','2014-12')).rename_vars({
                            "t2m": "tas",})
ERA5['weights']         = ERA5['tas'][0]*0 + np.cos(np.deg2rad(ERA5.lat))
lat = ERA5.lat
lon = ERA5.lon
reference_grid = ERA5['tas'].isel(time=0).load()

raw_weights     = np.cos(np.deg2rad(lat.values))[:, None] * np.ones(reference_grid.shape)
raw_weights     = np.where(np.isfinite(reference_grid.values), raw_weights, 0.)
global_weights  = raw_weights / raw_weights.sum()

In [14]:
# Only this observation-overlap window is processed. Month-start dates below are
# shared labels, not a conversion of model calendars or an interpolation in time.
overlap_data_root = Path("/scratch/leiff/MCA/amip_clean/data")
overlap_start, overlap_end, overlap_ddof = "2000-03", "2014-12", 1
overlap_months = np.arange(np.datetime64(overlap_start, "M"), np.datetime64(overlap_end, "M") + 1)
n_overlap       = len(overlap_months)
overlap_month_numbers = overlap_months.astype(int) % 12 + 1
overlap_lat_dim, overlap_lon_dim = "lat", "lon"
overlap_obs_paths = {
    "T": overlap_data_root / "ERA5/tas_194001-202412_g025.nc",
    "R": overlap_data_root / "CERES/CERES_net_200003-202604_g025.nc",
}

In [ ]:
tas_obs = detrend_monthly_fast(xr.open_dataset(overlap_obs_paths['T'])['t2m'].sel(time=slice(str(overlap_months[0]), str(overlap_months[-1]))).data)
N_obs = detrend_monthly_fast(xr.open_dataset(overlap_obs_paths['R'])['toa_net_all_mon'].sel(time=slice(str(overlap_months[0]), str(overlap_months[-1]))).data)

T_global_obs    = np.sum(tas_obs * global_weights, axis=(1, 2))
R_global_obs    = np.sum(N_obs   * global_weights, axis=(1, 2))

In [ ]:
plt.plot(T_global_obs)
plt.plot(R_global_obs)
print(stats.linregress(T_global_obs,R_global_obs))

In [15]:
full_start, full_end, full_ddof = "1870-01", "2014-12", 1
full_months = np.arange(np.datetime64(full_start, "M"), np.datetime64(full_end, "M") + 1)
full_month_numbers = full_months.astype(int) % 12 + 1
full_n_years    = np.int32(np.ceil(len(full_month_numbers)/12))

In [16]:
combined_keys = ["corr_Ti_Ri", "corr_T_Ri", "corr_Ti_R", "corr_T_R",
        "sigma_Ti",
        "sigma_Ri",
        "sigma_T", "sigma_R",
        "cov_Ti_R",
        "cov_Zi_R",
        "cov_T_Qi",
        "cov_T_Ri",
        "cov_Zi_Ri",
        "cov_Ti_Qi",
        "cov_Ti_Ri"]

In [ ]:
CESM_test_tas

In [ ]:
CESM_test_tas   = xr.open_dataset(overlap_data_root / 
"HIST/CESM2/tas_Amon_CESM2_historical_combined_187001-201412_g025.nc").sel(time=slice(str(full_months[0]), str(full_months[-1])))

CESM_test_N     = xr.open_dataset(overlap_data_root / 
"HIST/CESM2/N_Amon_CESM2_historical_combined_187001-201412_g025.nc").sel(time=slice(str(full_months[0]), str(full_months[-1])))

collected = {key: [] for key in combined_keys}
collected_years = {key: [] for key in combined_keys}
print(collected)

for i in np.arange(full_n_years-14):
# for i in np.arange(2):
    e = 0
    # Note: the full time period starts in jan by default
    first_start = np.datetime64('1870-03', "M")
    print(first_start+12*i)
    # kill
    
    Ti      = detrend_monthly_fast(CESM_test_tas['tas'].isel(member=e).isel(time=slice(12*i,12*i + n_overlap)).data)
    Ri      = detrend_monthly_fast(CESM_test_N['N'].isel(member=e).isel(time=slice(12*i,12*i + n_overlap)).data)
    if Ti.shape != Ri.shape:
        raise ValueError("detrend_monthly_fast changed the input shape.")
    Ti = Ti - Ti.mean(axis=0)
    Ri = Ri - Ri.mean(axis=0)

    # Global detrended series are weighted averages of these same processed fields.
    # This is the convention you already checked against detrending global averages.
    T = np.sum(Ti * global_weights, axis=(1, 2))
    R = np.sum(Ri * global_weights, axis=(1, 2))
    T, R = T - T.mean(), R - R.mean()

    n_time  = len(T)
    ddof = 1

    # All four sigmas are TEMPORAL SDs, not spatial SDs of a map. Using the same
    # n_time - ddof divisor for each covariance makes the reconstruction exact.
    sigma_Ti, sigma_Ri = Ti.std(axis=0, ddof=ddof), Ri.std(axis=0, ddof=ddof)
    sigma_T, sigma_R = T.std(ddof=ddof), R.std(ddof=ddof)
    cov_Ti_Ri = np.sum(Ti * Ri, axis=0) / (n_time - ddof)
    cov_T_Ri = np.sum(T[:, None, None] * Ri, axis=0) / (n_time - ddof)
    cov_Ti_R = np.sum(Ti * R[:, None, None], axis=0) / (n_time - ddof)
    cov_T_R = np.sum(T * R) / (n_time - ddof)

    # Pearson correlation = covariance / (sigma_A * sigma_B). Undefined constant
    # series remain NaN; they are not interpreted as zero correlation. Area weights
    # enter the global indices above, not an additional weighting of local maps.
    corr_Ti_Ri = np.divide(cov_Ti_Ri, sigma_Ti * sigma_Ri, out=np.full_like(sigma_Ti, np.nan),
                           where=(sigma_Ti > 0) & (sigma_Ri > 0))
    corr_T_Ri = np.divide(cov_T_Ri, sigma_T * sigma_Ri, out=np.full_like(sigma_Ri, np.nan),
                          where=(sigma_T > 0) & (sigma_Ri > 0))
    corr_Ti_R = np.divide(cov_Ti_R, sigma_Ti * sigma_R, out=np.full_like(sigma_Ti, np.nan),
                          where=(sigma_Ti > 0) & (sigma_R > 0))
    corr_T_R = cov_T_R / (sigma_T * sigma_R) if sigma_T > 0 and sigma_R > 0 else np.nan

    # Local temperature versus global radiation.
    cov_Zi_R    = np.divide(cov_Ti_R, sigma_Ti, out=np.full_like(sigma_Ti, np.nan), where=(sigma_Ti > 0))
    # Global temperature versus local radiation.
    cov_T_Qi    = np.divide(cov_T_Ri, sigma_Ri, out=np.full_like(sigma_Ri, np.nan), where=(sigma_Ri > 0))
    # Local temperature versus local radiation: either term or both standardized.
    cov_Zi_Ri   = np.divide(cov_Ti_Ri, sigma_Ti, out=np.full_like(sigma_Ti, np.nan), where=(sigma_Ti > 0))
    cov_Ti_Qi   = np.divide(cov_Ti_Ri, sigma_Ri, out=np.full_like(sigma_Ri, np.nan), where=(sigma_Ri > 0))

    # Save values under standard keys
    here = {
        "corr_Ti_Ri": np.asarray(np.clip(corr_Ti_Ri, -1., 1.)),
        "corr_T_Ri": np.asarray(np.clip(corr_T_Ri, -1., 1.)),
        "corr_Ti_R": np.asarray(np.clip(corr_Ti_R, -1., 1.)),
        "corr_T_R": np.asarray(np.clip(corr_T_R, -1., 1.)),
        "sigma_Ti": np.asarray(sigma_Ti),
        "sigma_Ri": np.asarray(sigma_Ri),
        "sigma_T": np.asarray(sigma_T), "sigma_R": np.asarray(sigma_R),

        # Mine
        "cov_Ti_R": np.asarray(cov_Ti_R),
        "cov_Zi_R": np.asarray(cov_Zi_R),
        "cov_T_Qi": np.asarray(cov_T_Qi),
        "cov_T_Ri": np.asarray(cov_T_Ri),
        "cov_Zi_Ri": np.asarray(cov_Zi_Ri),
        "cov_Ti_Qi": np.asarray(cov_Ti_Qi),
        "cov_Ti_Ri": np.asarray(cov_Ti_Ri),
    }

    # collected.append(here)
    for key, value in here.items():
        collected[key].append(value)
    

    # plt.plot(T_global_here)
    # plt.plot(R_global_here)
    # print(stats.linregress(T_global_here,R_global_here))
    # kill

for key, value in collected.items():
    print(key)
    # print(np.asarray(value))
    collected_years[key].append(np.asarray(value))
    # stop


# T_raw   = np.array(CESM_test_tas.values)

# T_raw, R_raw = np.asarray(T_raw, dtype=float), np.asarray(R_raw, dtype=float)
# n_time = len(month_numbers)




In [ ]:
collected_years['corr_Ti_Ri']

In [ ]:
plt.plot()

In [ ]:
np.array(CESM_test_tas['tas'].data)

In [ ]:
full_month_numbers

In [ ]:
CESM_test_tas

In [ ]:
combined_keys = ["corr_Ti_Ri", "corr_T_Ri", "corr_Ti_R", "corr_T_R",
        "sigma_Ti",
        "sigma_Ri",
        "sigma_T", "sigma_R",
        "cov_Ti_R",
        "cov_Zi_R",
        "cov_T_Qi",
        "cov_T_Ri",
        "cov_Zi_Ri",
        "cov_Ti_Qi",
        "cov_Ti_Ri"]

In [17]:
# %% 1. Register your quantities and create EMPTY output structures
# In the supplied notebook, the preceding helper cell defines these functions.
# Elsewhere, import them from diagnostic_archive_helpers.py (put it on your path):
# from diagnostic_archive_helpers import (setup_diagnostic_archive, store_diagnostic_members,
#     copy_diagnostic_subset, aggregate_diagnostic_archive, save_diagnostic_archive, load_diagnostic_archive)
import numpy as np
from pathlib import Path

# Each quantity has its OWN per-member shape. These two placeholders demonstrate
# a map and scalar; the commented entries show a series and higher-dimensional array.
# dims excludes the member axis. shape is optional; otherwise the first stored
# model determines the shape required for that quantity everywhere in this archive.



combined_keys = ["corr_Ti_Ri", "corr_T_Ri", "corr_Ti_R", "corr_T_R", "sigma_Ti",
        "sigma_Ri", "sigma_T", "sigma_R", "cov_Ti_R", "cov_Zi_R", "cov_T_Qi",
        "cov_T_Ri", "cov_Zi_Ri", "cov_Ti_Qi", "cov_Ti_Ri"]

# combined_keys = ["corr_T_R", "sigma_Ti", "sigma_Ri", "sigma_T", "sigma_R", "cov_Zi_R", "cov_Zi_Ri"]

quantity_info = {
    "corr_Ti_Ri": {"kind": "array", "dims": ("year", "lat", "lon"), "units": "",
                 "meaning": "...", "apply_mask": False},
    "corr_T_Ri": {"kind": "array", "dims": ("year", "lat", "lon"), "units": "",
                 "meaning": "...", "apply_mask": False},
    "corr_Ti_R": {"kind": "array", "dims": ("year", "lat", "lon"), "units": "",
                 "meaning": "...", "apply_mask": False},
    "corr_T_R": {"kind": "series", "dims": ("year",), "units": "",
                 "meaning": "...", "apply_mask": False},
    "sigma_Ti": {"kind": "array", "dims": ("year", "lat", "lon"), "units": "",
                 "meaning": "...", "apply_mask": False},
    "sigma_Ri": {"kind": "array", "dims": ("year", "lat", "lon"), "units": "",
                 "meaning": "...", "apply_mask": False},
    "sigma_T": {"kind": "series", "dims": ("year",), "units": "",
                 "meaning": "...", "apply_mask": False},
    "sigma_R": {"kind": "series", "dims": ("year",), "units": "",
                 "meaning": "...", "apply_mask": False},
    "cov_Ti_R": {"kind": "array", "dims": ("year", "lat", "lon"), "units": "Wm-2K",
                 "meaning": "...", "apply_mask": False},
    "cov_Zi_R": {"kind": "array", "dims": ("year", "lat", "lon"), "units": "Wm-2",
                 "meaning": "...", "apply_mask": False},
    "cov_T_Qi": {"kind": "array", "dims": ("year", "lat", "lon"), "units": "K",
                 "meaning": "...", "apply_mask": False},
    "cov_T_Ri": {"kind": "array", "dims": ("year", "lat", "lon"), "units": "Wm-2K",
                 "meaning": "...", "apply_mask": False},
    "cov_Zi_Ri": {"kind": "array", "dims": ("year", "lat", "lon"), "units": "Wm-2",
                 "meaning": "...", "apply_mask": False},
    "cov_Ti_Qi": {"kind": "array", "dims": ("year", "lat", "lon"), "units": "K",
                 "meaning": "...", "apply_mask": False},
    "cov_Ti_Ri": {"kind": "array", "dims": ("year", "lat", "lon"), "units": "Wm-2K",
                 "meaning": "...", "apply_mask": False},
    
}

# Reuse the core archive's explicit experiment/model/member inventory. For a new
# project, supply your own dictionaries in the same format. Coordinates are labels
# and validation aids, not instructions to regrid or align arbitrary arrays.
experiments = ["historical", "historical_matched", "amip_hist"]
coordinates = {key: ERA5[key] for key in ["lat", "lon", "time"]}
# Each label identifies the year in which the window begins in March.
coordinates["year"] = 1870 + np.arange(full_n_years - 14)
print(coordinates)
# coordinates["lag"] = np.arange(-12, 13)     # If you add a lag-indexed quantity.
analysis_mask = None                         # Optional 2-D Boolean or 1/NaN mask.
# analysis_mask = overlap_coordinates["local_support"]
# A mask affects only the FINAL STORED arrays of apply_mask=True fields.
# If you need region-specific calculations (e.g. a regional time series), apply
# the region to your INPUTS and weights yourself in the calculation block below.

variability_archive_dir = Path("/scratch/leiff/MCA/amip_clean/data/maps_2000_2014_covImp/overlap_core_200003_201412/")
overlap_model_names = np.load(variability_archive_dir / "overlap_model_names.npy", allow_pickle=True).item()
overlap_member_ids = np.load(variability_archive_dir / "overlap_member_ids.npy", allow_pickle=True).item()

diagnostics = setup_diagnostic_archive(
    quantity_info, overlap_model_names, overlap_member_ids,
    experiments=experiments, coordinates=coordinates, mask=analysis_mask,
    save=True, 
    savepath=variability_archive_dir / "stationarity" / "march1870_n178_historical.npy", 
    overwrite=False,
    metadata={
        "title": "178-month maps from every march", "purpose": "Look at all observation overlap length periods starting at each month N=131",
        "period": "1870-03 through 2014-12", "source_archive": str(variability_archive_dir),
        "preprocessing": "Detrended/deseasonalized within each 178-month period",
        "formula": "Standard cov diagnostics",
        "notes": "FILL IN: units, signs, missing-value rules, interpretation/caveats",
    },
)
# Optional extra dictionaries/arrays are also saved in the same archive; helpers
# leave them alone. No spatial weights are applied automatically to your results.
# diagnostics["weights"] = np.asarray(overlap_coordinates["base_weights"]).copy()
# diagnostics["observations"] = {}       # Fill with your own reference diagnostics if useful.

# Optional reuse: only skip a subset if its source is ALSO selected. This is valid
# for member diagnostics independent of which other models belong to the group.
# Set False if your calculations themselves depend on the experiment's population.
reuse_historical_matched = False
reuse_matched = reuse_historical_matched and {"historical", "historical_matched"}.issubset(diagnostics["experiments"])
calculate_experiments = [e for e in diagnostics["experiments"] if not (reuse_matched and e == "historical_matched")]


{'lat': <xarray.DataArray 'lat' (lat: 72)> Size: 576B
array([-88.75, -86.25, -83.75, -81.25, -78.75, -76.25, -73.75, -71.25, -68.75,
       -66.25, -63.75, -61.25, -58.75, -56.25, -53.75, -51.25, -48.75, -46.25,
       -43.75, -41.25, -38.75, -36.25, -33.75, -31.25, -28.75, -26.25, -23.75,
       -21.25, -18.75, -16.25, -13.75, -11.25,  -8.75,  -6.25,  -3.75,  -1.25,
         1.25,   3.75,   6.25,   8.75,  11.25,  13.75,  16.25,  18.75,  21.25,
        23.75,  26.25,  28.75,  31.25,  33.75,  36.25,  38.75,  41.25,  43.75,
        46.25,  48.75,  51.25,  53.75,  56.25,  58.75,  61.25,  63.75,  66.25,
        68.75,  71.25,  73.75,  76.25,  78.75,  81.25,  83.75,  86.25,  88.75])
Coordinates:
  * lat      (lat) float64 576B -88.75 -86.25 -83.75 ... 83.75 86.25 88.75
Attributes:
    standard_name:  latitude
    long_name:      latitude
    units:          degrees_north
    axis:           Y
    bounds:         lat_bnds, 'lon': <xarray.DataArray 'lon' (lon: 144)> Size: 1kB
array([  1.25,  

In [18]:
from pathlib import Path
# %% 2. YOUR calculations inside the visible experiment -> model -> member loops
# This cell deliberately stops until you fill 'here'. Nothing scientific is
# calculated automatically. Rerun setup when changing labels/quantities/masks;
# rerunning it resets ONLY this new diagnostics archive, not your source archive.
for experiment in calculate_experiments:
    for model in diagnostics["model_names"][experiment]:
        basepath = '/scratch/leiff/MCA/amip_clean/data/maps_2000_2014_covImp/overlap_core_200003_201412/stationarity/'
        
        if Path(basepath + f"march_n178_{experiment}_{model}_member_values.npy").exists():
            continue  # This was already processed

        member_labels = diagnostics["member_ids"][experiment][model]
        collected = {quantity: [] for quantity in diagnostics["quantity_info"]}
        print(f"{experiment} / {model}: {len(member_labels)} members")

        # OPTIONAL MODEL-LEVEL PREPARATION: open files or retrieve all members once.
        # With raw files, check their member labels against member_labels; reorder
        # explicitly if needed. Close any files you open after this model's loop.
        # model_input = overlap_member_values[experiment]["sigma_Ti"][model]
        #####
        if experiment=='historical' or experiment=='historical_matched':
            model_path  = 'HIST'
            exp_name = 'historical'
        elif experiment=='amip_hist':
            model_path = 'AMIP'
            exp_name = 'amip-hist'

        time_window = slice(str(full_months[0])[:7], str(full_months[-1])[:7])
        path_tas    = overlap_data_root / f"{model_path}/{model}/tas_Amon_{model}_{exp_name}_combined_187001-201412_g025.nc"
        path_N      = overlap_data_root / f"{model_path}/{model}/N_Amon_{model}_{exp_name}_combined_187001-201412_g025.nc"
        

        # Open once per model. Keep all file-dependent calculations inside this block.
        with xr.open_dataset(path_tas) as file_tas, xr.open_dataset(path_N) as file_N:
            ds_tas  = file_tas.sel(time=time_window)
            ds_N    = file_N.sel(time=time_window)

            for member_index, member_id in enumerate(member_labels):
                collected_sub = {key: [] for key in combined_keys}
                collected_years = {key: [] for key in combined_keys}

                # Read this member once; reuse its raw data for every rolling window.
                tas_member = ds_tas["tas"].isel(member=member_index).to_numpy()
                N_member = ds_N["N"].isel(member=member_index).to_numpy()

                for i in np.arange(full_n_years-14):
                    # Note: the full time period starts in jan by default
                    # first_start = np.datetime64('1870-03', "M")

                    window = slice(12*i + 2, 12*i + 2 + n_overlap) # +2 to start in March
                    # Copies protect overlapping windows if detrending modifies its input.
                    Ti = detrend_monthly_fast(tas_member[window].copy())
                    Ri = detrend_monthly_fast(N_member[window].copy())

                    if Ti.shape != Ri.shape:
                        raise ValueError("Ti.shape != Ri.shape")
                    Ti = Ti - Ti.mean(axis=0)
                    Ri = Ri - Ri.mean(axis=0)

                    # Global detrended series are weighted averages of these same processed fields.
                    # T = np.sum(Ti * global_weights, axis=(1, 2))
                    # R = np.sum(Ri * global_weights, axis=(1, 2))
                    T = np.einsum("tij,ij->t", Ti, global_weights)
                    R = np.einsum("tij,ij->t", Ri, global_weights)
                    T, R = T - T.mean(), R - R.mean()

                    n_time  = len(T)
                    ddof = 1

                    # All four sigmas are TEMPORAL SDs, not spatial SDs of a map. Using the same
                    # n_time - ddof divisor for each covariance makes the reconstruction exact.
                    sigma_Ti, sigma_Ri = Ti.std(axis=0, ddof=ddof), Ri.std(axis=0, ddof=ddof)
                    sigma_T, sigma_R = T.std(ddof=ddof), R.std(ddof=ddof)
                    # cov_Ti_Ri = np.sum(Ti * Ri, axis=0) / (n_time - ddof)
                    # cov_T_Ri = np.sum(T[:, None, None] * Ri, axis=0) / (n_time - ddof)
                    # cov_Ti_R = np.sum(Ti * R[:, None, None], axis=0) / (n_time - ddof)
                    cov_Ti_Ri = np.einsum("tij,tij->ij", Ti, Ri) / (n_time - ddof)
                    cov_T_Ri = np.einsum("t,tij->ij", T, Ri) / (n_time - ddof)
                    cov_Ti_R = np.einsum("tij,t->ij", Ti, R) / (n_time - ddof)
                    cov_T_R = np.sum(T * R) / (n_time - ddof)

                    # Pearson correlation = covariance / (sigma_A * sigma_B). Area weights
                    # enter the global indices above, not an additional weighting of local maps.
                    corr_Ti_Ri = np.divide(cov_Ti_Ri, sigma_Ti * sigma_Ri, out=np.full_like(sigma_Ti, np.nan),
                                        where=(sigma_Ti > 0) & (sigma_Ri > 0))
                    corr_T_Ri = np.divide(cov_T_Ri, sigma_T * sigma_Ri, out=np.full_like(sigma_Ri, np.nan),
                                        where=(sigma_T > 0) & (sigma_Ri > 0))
                    corr_Ti_R = np.divide(cov_Ti_R, sigma_Ti * sigma_R, out=np.full_like(sigma_Ti, np.nan),
                                        where=(sigma_Ti > 0) & (sigma_R > 0))
                    corr_T_R = cov_T_R / (sigma_T * sigma_R) if sigma_T > 0 and sigma_R > 0 else np.nan

                    # Local temperature versus global radiation.
                    cov_Zi_R    = np.divide(cov_Ti_R, sigma_Ti, out=np.full_like(sigma_Ti, np.nan), where=(sigma_Ti > 0))
                    # Global temperature versus local radiation.
                    cov_T_Qi    = np.divide(cov_T_Ri, sigma_Ri, out=np.full_like(sigma_Ri, np.nan), where=(sigma_Ri > 0))
                    # Local temperature versus local radiation: either term or both standardized.
                    cov_Zi_Ri   = np.divide(cov_Ti_Ri, sigma_Ti, out=np.full_like(sigma_Ti, np.nan), where=(sigma_Ti > 0))
                    cov_Ti_Qi   = np.divide(cov_Ti_Ri, sigma_Ri, out=np.full_like(sigma_Ri, np.nan), where=(sigma_Ri > 0))

                    # Save values under standard keys
                    here_in = {
                        "corr_Ti_Ri": np.asarray(np.clip(corr_Ti_Ri, -1., 1.)),
                        "corr_T_Ri": np.asarray(np.clip(corr_T_Ri, -1., 1.)),
                        "corr_Ti_R": np.asarray(np.clip(corr_Ti_R, -1., 1.)),
                        "corr_T_R": np.asarray(np.clip(corr_T_R, -1., 1.)),
                        "sigma_Ti": np.asarray(sigma_Ti),
                        "sigma_Ri": np.asarray(sigma_Ri),
                        "sigma_T": np.asarray(sigma_T), "sigma_R": np.asarray(sigma_R),

                        "cov_Ti_R": np.asarray(cov_Ti_R),
                        "cov_Zi_R": np.asarray(cov_Zi_R),
                        "cov_T_Qi": np.asarray(cov_T_Qi),
                        "cov_T_Ri": np.asarray(cov_T_Ri),
                        "cov_Zi_Ri": np.asarray(cov_Zi_Ri),
                        "cov_Ti_Qi": np.asarray(cov_Ti_Qi),
                        "cov_Ti_Ri": np.asarray(cov_Ti_Ri),
                    }

                    # Stack up values for each year start
                    for key, value in here_in.items():
                        collected_sub[key].append(value)

                # Combine into a single keyed dict with arrays as values
                for key, value in collected_sub.items():
                    collected_years[key] = np.asarray(value)
                here = collected_years
                # Clear up space
                del tas_member, N_member

                # Require every field explicitly. An all-NaN numeric result is allowed
                # when scientifically undefined; an accidentally missing field is not.
                if set(here) != set(diagnostics["quantity_info"]):
                    raise ValueError("Fill 'here' with exactly the registered quantity keys before running this loop.")
                for quantity in diagnostics["quantity_info"]:
                    collected[quantity].append(here[quantity])

            collected_members = {keys: [] for keys in collected.keys()}
            for key in collected.keys():
                collected_members[key]  = np.stack(collected[key], axis=0)

            
            np.save(basepath + f"march_n178_{experiment}_{model}_member_values.npy", collected_members, allow_pickle=True)

            del collected_members, collected, collected_years, here
            # print()
            # kill
                
        # Close/release your raw model input here if you opened one above.

# Reuse full historical results for the matched model/member inventory. Actual
# member IDs are checked and reordered if necessary; copied arrays are independent.
if reuse_matched:
    copy_diagnostic_subset(diagnostics, source="historical", target="historical_matched", overwrite=True)


In [11]:
experiments = ["historical", "historical_matched", "amip_hist"]

basepath = '/scratch/leiff/MCA/amip_clean/data/maps_2000_2014_covImp/overlap_core_200003_201412/stationarity/'
test = np.load(basepath + f"march_n178_historical_ACCESS-CM2_member_values.npy",allow_pickle=True).item()

In [13]:
test['corr_Ti_Ri']

array([[[[-0.92174554, -0.9219327 , -0.92206978, ..., -0.91624674,
          -0.91549792, -0.91764544],
         [-0.93562899, -0.93738448, -0.93836971, ..., -0.9249158 ,
          -0.92935899, -0.93199904],
         [-0.90624656, -0.91134352, -0.91773506, ..., -0.88671139,
          -0.89254863, -0.89926019],
         ...,
         [-0.50964181, -0.5259853 , -0.54076975, ..., -0.43315402,
          -0.45853282, -0.48564147],
         [-0.48110441, -0.49225574, -0.50072132, ..., -0.46798658,
          -0.47048036, -0.47314581],
         [-0.4667615 , -0.46700747, -0.46696399, ..., -0.46630039,
          -0.46662384, -0.46413527]],

        [[-0.92536076, -0.9255937 , -0.92564054, ..., -0.92018114,
          -0.91989388, -0.92170563],
         [-0.93531629, -0.93748069, -0.93850097, ..., -0.92512335,
          -0.92916959, -0.93166388],
         [-0.90682282, -0.91223787, -0.91887568, ..., -0.8874727 ,
          -0.89288323, -0.89959857],
         ...,
         [-0.58748676, -0.60567466